# JPMorganChase Quantitative Modeling Tech Screen — Part 2
## Systematic Data Inspection, Cleaning, and Verification

This notebook is designed for a **Model Risk Governance Review / applied data science tech screen**.

The core workflow is:

> **UNDERSTAND → INSPECT → IDENTIFY ISSUES → CLEAN → VERIFY → DOCUMENT ASSUMPTIONS**

The goal is not just to make the dataset usable. The goal is to show that you can **challenge the data, justify transformations, and verify that your cleaning did not introduce new problems**.

---

### What interviewers are likely scoring

1. Can you inspect an unfamiliar dataset quickly?
2. Can you find missing values, duplicates, bad types, impossible values, and suspicious outliers?
3. Can you choose a defensible cleaning method?
4. Can you explain the assumption behind that method?
5. Can you verify the result after cleaning?
6. Can you keep the code simple and runnable?

---

### Best interview habit

For every transformation, explicitly think:

1. **What is wrong?**
2. **Why might it be wrong?**
3. **What transformation am I applying?**
4. **What assumption does that transformation introduce?**
5. **How will I verify the result afterward?**

# 0. Setup

We create a small synthetic loan dataset containing realistic data-quality problems:

- missing values
- duplicate rows
- numeric columns stored as strings
- zero and negative income
- implausible loan amounts
- percentage values represented inconsistently
- extreme ratio values
- missing time-series observations

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

raw = {
    "customer_id": [101,102,103,104,105,106,107,108,109,110,110],
    "date": [
        "2026-01-01","2026-01-01","2026-01-02","2026-01-02","2026-01-03",
        "2026-01-03","2026-01-04","2026-01-04","2026-01-05","2026-01-05","2026-01-05"
    ],
    "income": ["60000","72000",None,"0","85000","-5000","95,000","110000","78000","82000","82000"],
    "loan_amount": [20000,30000,25000,45000,40000,15000,50000,900000,28000,31000,31000],
    "asset_value": [100000,120000,110000,150000,180000,90000,200000,250000,130000,140000,140000],
    "debt_payment": [900,1200,1000,1600,None,500,1300,2000,950,1100,1100],
    "utilization": [0.25,0.40,40,0.55,0.35,0.30,None,0.85,0.50,0.45,0.45],
    "credit_score": [720,680,700,640,750,610,690,730,None,670,670],
    "default": [0,0,0,1,0,1,0,0,0,1,1]
}

df = pd.DataFrame(raw)
df

,customer_id,date,income,loan_amount,asset_value,debt_payment,utilization,credit_score,default
0,101,2026-01-01,60000,20000,100000,900.0,0.25,720.0,0
1,102,2026-01-01,72000,30000,120000,1200.0,0.40,680.0,0
2,103,2026-01-02,NaN,25000,110000,1000.0,40.00,700.0,0
3,104,2026-01-02,0,45000,150000,1600.0,0.55,640.0,1
4,105,2026-01-03,85000,40000,180000,NaN,0.35,750.0,0
5,106,2026-01-03,-5000,15000,90000,500.0,0.30,610.0,1
6,107,2026-01-04,"95,000",50000,200000,1300.0,NaN,690.0,0
7,108,2026-01-04,110000,900000,250000,2000.0,0.85,730.0,0
8,109,2026-01-05,78000,28000,130000,950.0,0.50,NaN,0
9,110,2026-01-05,82000,31000,140000,1100.0,0.45,670.0,1


# 1. Understand the dataset before touching it

## What to inspect

Before cleaning, identify:

- **unit of observation** — one row per borrower? loan? borrower-month?
- **target variable** — what are we predicting?
- **important financial variables**
- **expected ranges / units**
- whether the data is cross-sectional or time-ordered

Do not start changing values until you understand what each column is supposed to represent.

### What to say in the interview

> "Before making any transformations, I want to understand the unit of observation, the target variable, and the meaning and units of the main features. That determines what values are plausible and which cleaning methods are defensible."

In [3]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst rows:")
print(df.head())

print("\nTarget distribution:")
print(df["default"].value_counts(dropna=False))

Shape: (11, 9)

Columns:
['customer_id', 'date', 'income', 'loan_amount', 'asset_value', 'debt_payment', 'utilization', 'credit_score', 'default']

First rows:
   customer_id        date income  loan_amount  asset_value  debt_payment  utilization  credit_score  default
0          101  2026-01-01  60000        20000       100000         900.0         0.25         720.0        0
1          102  2026-01-01  72000        30000       120000        1200.0         0.40         680.0        0
2          103  2026-01-02    NaN        25000       110000        1000.0        40.00         700.0        0
3          104  2026-01-02      0        45000       150000        1600.0         0.55         640.0        1
4          105  2026-01-03  85000        40000       180000           NaN         0.35         750.0        0

Target distribution:
default
0    7
1    4
Name: count, dtype: int64


# 2. Initial structural inspection

This is the first block of code you should almost always run.

### What to say

> "I'll start with basic structural checks: shape, data types, missingness, and summary statistics. This usually surfaces most obvious data-quality issues very quickly."

In [4]:
print("=== SHAPE ===")
print(df.shape)

print("\n=== DTYPES ===")
print(df.dtypes)

print("\n=== INFO ===")
df.info()

print("\n=== MISSING COUNTS ===")
print(df.isna().sum())

print("\n=== MISSING RATES ===")
print(df.isna().mean().sort_values(ascending=False))

print("\n=== NUMERIC SUMMARY ===")
print(df.describe(include="all"))

=== SHAPE ===
(11, 9)

=== DTYPES ===
customer_id       int64
date                str
income              str
loan_amount       int64
asset_value       int64
debt_payment    float64
utilization     float64
credit_score    float64
default           int64
dtype: object

=== INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   customer_id   11 non-null     int64  
 1   date          11 non-null     str    
 2   income        10 non-null     str    
 3   loan_amount   11 non-null     int64  
 4   asset_value   11 non-null     int64  
 5   debt_payment  10 non-null     float64
 6   utilization   10 non-null     float64
 7   credit_score  10 non-null     float64
 8   default       11 non-null     int64  
dtypes: float64(3), int64(4), str(2)
memory usage: 924.0 bytes

=== MISSING COUNTS ===
customer_id     0
date            0
income          1
loan_amount  

## What we should notice here

Potential problems:

- `income` is stored as `object`, not numeric.
- `income` contains commas and missing values.
- there is at least one zero / negative income.
- `utilization` contains both decimals like `0.40` and a suspicious value `40`.
- `loan_amount` has a very large value (`900000`).
- there is a duplicate row.
- some important predictors are missing.

This is the point where you should **form hypotheses**, not immediately delete rows.

# 3. Check duplicates

Duplicates can distort model fitting and summary statistics.

### Important nuance

A duplicate is not automatically an error. For example, repeated transactions may be legitimate. You need to interpret duplication relative to the dataset's unit of observation.

### What to say

> "I'll check for exact duplicates first. If the dataset is supposed to contain one row per borrower-date, I would also check duplicates on that business key."

In [7]:
print("Exact duplicate rows:", df.duplicated().sum())

print("\nDuplicate customer/date keys:")
print(df[df.duplicated(subset=["customer_id", "date"], keep=False)])

Exact duplicate rows: 1

Duplicate customer/date keys:
    customer_id        date income  loan_amount  asset_value  debt_payment  utilization  credit_score  default
9           110  2026-01-05  82000        31000       140000        1100.0         0.45         670.0        1
10          110  2026-01-05  82000        31000       140000        1100.0         0.45         670.0        1


In [8]:
# For this synthetic example, the last row is an accidental duplicate.
df_clean = df.drop_duplicates().copy()

print("Rows before:", len(df))
print("Rows after :", len(df_clean))

Rows before: 11
Rows after : 10


# 4. Fix data types carefully

A common interview issue is that numeric data arrives as strings because of commas, currency symbols, or malformed entries.

Use `pd.to_numeric(..., errors="coerce")` when you want invalid entries to become `NaN` rather than crashing.

### What to say

> "This column should be numeric, so I'll standardize the formatting and coerce anything that still cannot be parsed to missing. Then I'll inspect how many new missing values that conversion created."

In [9]:
# Remove thousands separators before numeric conversion.
df_clean["income"] = (
    df_clean["income"]
    .str.replace(",", "", regex=False)
)

before_missing = df_clean["income"].isna().sum()

df_clean["income"] = pd.to_numeric(df_clean["income"], errors="coerce")

after_missing = df_clean["income"].isna().sum()

print("Income dtype:", df_clean["income"].dtype)
print("Missing before conversion:", before_missing)
print("Missing after conversion :", after_missing)
print(df_clean["income"])

Income dtype: float64
Missing before conversion: 1
Missing after conversion : 1
0     60000.0
1     72000.0
2         NaN
3         0.0
4     85000.0
5     -5000.0
6     95000.0
7    110000.0
8     78000.0
9     82000.0
Name: income, dtype: float64


In [10]:
# Parse dates explicitly.
df_clean["date"] = pd.to_datetime(df_clean["date"], errors="coerce")

print(df_clean.dtypes)

customer_id              int64
date            datetime64[us]
income                 float64
loan_amount              int64
asset_value              int64
debt_payment           float64
utilization            float64
credit_score           float64
default                  int64
dtype: object


# 5. Validate ranges and business plausibility

This is one of the most important steps for a Model Risk interview.

Do not assume a value is valid simply because it is numeric.

Ask:

- Can income be zero or negative?
- Can a utilization ratio be 40?
- Can an LTV be greater than 1? Sometimes yes, but perhaps not by huge amounts.
- Is a $900,000 loan plausible relative to the rest of this portfolio?
- Are percentages represented as `40` or `0.40`?

### What to say

> "Now that the types are correct, I want to separate statistical outliers from economically impossible or suspicious values. Those require different treatment."

In [11]:
numeric_cols = [
    "income", "loan_amount", "asset_value",
    "debt_payment", "utilization", "credit_score"
]

print(df_clean[numeric_cols].describe().T)

              count           mean            std       min        25%        50%        75%       max
income          9.0   64111.111111   40272.336797  -5000.00   60000.00   78000.00   85000.00  110000.0
loan_amount    10.0  118400.000000  274840.236420  15000.00   25750.00   30500.00   43750.00  900000.0
asset_value    10.0  147000.000000   49899.899799  90000.00  112500.00  135000.00  172500.00  250000.0
debt_payment    9.0    1172.222222     432.370726    500.00     950.00    1100.00    1300.00    2000.0
utilization     9.0       4.850000      13.182422      0.25       0.35       0.45       0.55      40.0
credit_score    9.0     687.777778      44.095855    610.00     670.00     690.00     720.00     750.0


In [12]:
print("Income <= 0:")
print(df_clean.loc[df_clean["income"] <= 0,
                   ["customer_id", "income"]])

print("\nUtilization > 1:")
print(df_clean.loc[df_clean["utilization"] > 1,
                   ["customer_id", "utilization"]])

print("\nLargest loan amounts:")
print(df_clean.nlargest(5, "loan_amount")[
    ["customer_id", "loan_amount", "income", "asset_value"]
])

Income <= 0:
   customer_id  income
3          104     0.0
5          106 -5000.0

Utilization > 1:
   customer_id  utilization
2          103         40.0

Largest loan amounts:
   customer_id  loan_amount    income  asset_value
7          108       900000  110000.0       250000
6          107        50000   95000.0       200000
3          104        45000       0.0       150000
4          105        40000   85000.0       180000
9          110        31000   82000.0       140000


# 6. Correct inconsistent units only when evidence supports it

Suppose `utilization` should be a fraction between 0 and 1.

A value of `40` is likely representing **40%**, while the rest of the column uses decimal representation.

We can convert values greater than 1 to percentage form — but only if the problem statement or data pattern supports that assumption.

### What to say

> "This looks like a unit inconsistency rather than a genuine outlier. Most rows use decimal utilization, while this value appears to be stored as a percentage. I would confirm the field definition if possible; given the pattern, I'll standardize it to decimal form."

In [13]:
mask_pct = df_clean["utilization"] > 1

print("Rows affected:")
print(df_clean.loc[mask_pct, ["customer_id", "utilization"]])

df_clean.loc[mask_pct, "utilization"] = (
    df_clean.loc[mask_pct, "utilization"] / 100
)

print("\nAfter standardization:")
print(df_clean["utilization"])

Rows affected:
   customer_id  utilization
2          103         40.0

After standardization:
0    0.25
1    0.40
2    0.40
3    0.55
4    0.35
5    0.30
6     NaN
7    0.85
8    0.50
9    0.45
Name: utilization, dtype: float64


# 7. Handle impossible denominator values before computing ratios

This is a very important interview pattern.

If you calculate:

`loan_amount / income`

and income is zero or negative, you can create huge or meaningless numbers.

Do **not** jump directly to scaling.

First verify the denominator.

### What to say

> "Before calculating ratios, I want to validate the denominator. A zero or negative denominator can create extreme values that look like outliers but are really data-quality problems."

In [14]:
print(df_clean.loc[df_clean["income"] <= 0,
                   ["customer_id", "income", "loan_amount"]])

   customer_id  income  loan_amount
3          104     0.0        45000
5          106 -5000.0        15000


In [15]:
# Treat non-positive income as invalid for this ratio.
df_clean.loc[df_clean["income"] <= 0, "income"] = np.nan

# Compute ratio safely.
df_clean["loan_to_income"] = np.where(
    df_clean["income"] > 0,
    df_clean["loan_amount"] / df_clean["income"],
    np.nan
)

print(df_clean[["customer_id", "income", "loan_amount", "loan_to_income"]])

   customer_id    income  loan_amount  loan_to_income
0          101   60000.0        20000        0.333333
1          102   72000.0        30000        0.416667
2          103       NaN        25000             NaN
3          104       NaN        45000             NaN
4          105   85000.0        40000        0.470588
5          106       NaN        15000             NaN
6          107   95000.0        50000        0.526316
7          108  110000.0       900000        8.181818
8          109   78000.0        28000        0.358974
9          110   82000.0        31000        0.378049


# 8. Missing values: inspect before imputing

Never start with `fillna()` blindly.

First ask:

- how much data is missing?
- is missingness concentrated in one variable?
- is there a temporal ordering?
- does the missing value itself carry information?
- would dropping rows bias the sample?

### What to say

> "Before imputing, I want to understand the missingness pattern. The appropriate method depends on whether the feature is cross-sectional, time-ordered, skewed, or potentially missing for a systematic reason."

In [16]:
missing_summary = pd.DataFrame({
    "missing_count": df_clean.isna().sum(),
    "missing_rate": df_clean.isna().mean()
}).sort_values("missing_rate", ascending=False)

missing_summary

,missing_count,missing_rate
income,3,0.3
loan_to_income,3,0.3
debt_payment,1,0.1
utilization,1,0.1
credit_score,1,0.1
customer_id,0,0.0
date,0,0.0
loan_amount,0,0.0
asset_value,0,0.0
default,0,0.0


# 9. Median imputation for skewed numeric variables

For a generic numeric feature with outliers, the median is often safer than the mean.

This is not universally correct — it is simply a defensible baseline.

### What to say

> "For this numeric variable, the median is more robust to extreme values than the mean. I would still check whether missingness itself is informative before removing that signal."

In [17]:
median_income = df_clean["income"].median()
print("Median income:", median_income)

df_clean["income_missing"] = df_clean["income"].isna().astype(int)
df_clean["income"] = df_clean["income"].fillna(median_income)

print(df_clean[["customer_id", "income", "income_missing"]])

Median income: 82000.0
   customer_id    income  income_missing
0          101   60000.0               0
1          102   72000.0               0
2          103   82000.0               1
3          104   82000.0               1
4          105   85000.0               0
5          106   82000.0               1
6          107   95000.0               0
7          108  110000.0               0
8          109   78000.0               0
9          110   82000.0               0


# 10. Forward fill / interpolation for ordered data

Your friend specifically remembered `ffill()` and interpolation.

The key is knowing **when** they are defensible.

## Forward fill

Use when:

- observations are ordered in time
- the previous value reasonably remains valid until updated

Example:

```python
df["x"] = df["x"].ffill()
```

## Linear interpolation

Use when:

- values evolve continuously
- neighboring observations contain information about the missing point

Example:

```python
df["x"] = df["x"].interpolate()
```

### What to say

> "I would only forward-fill a variable if carrying the last observed value forward has a valid business interpretation. Similarly, interpolation assumes continuity between neighboring observations, so I would not use it for arbitrary categorical or jump-driven variables."

In [18]:
# Small time-series example for practice.
ts = pd.DataFrame({
    "date": pd.date_range("2026-01-01", periods=6),
    "loan_ratio": [0.30, np.nan, np.nan, 0.42, np.nan, 0.48]
})

print("Original:")
print(ts)

ts["ffill"] = ts["loan_ratio"].ffill()
ts["interpolated"] = ts["loan_ratio"].interpolate()

print("\nComparison:")
print(ts)

Original:
        date  loan_ratio
0 2026-01-01        0.30
1 2026-01-02         NaN
2 2026-01-03         NaN
3 2026-01-04        0.42
4 2026-01-05         NaN
5 2026-01-06        0.48

Comparison:
        date  loan_ratio  ffill  interpolated
0 2026-01-01        0.30   0.30          0.30
1 2026-01-02         NaN   0.30          0.34
2 2026-01-03         NaN   0.30          0.38
3 2026-01-04        0.42   0.42          0.42
4 2026-01-05         NaN   0.42          0.45
5 2026-01-06        0.48   0.48          0.48


# 11. Outliers: investigate, do not automatically delete

A statistical outlier may still be a genuine observation.

In credit data, extreme borrowers may be exactly the observations that matter most.

Use tools like:

- quantiles
- IQR rule
- sorting extreme values
- comparing related features

### What to say

> "I would flag outliers first rather than automatically removing them. If an observation is a data error, I would correct or exclude it. If it is genuine, removing it could bias the model and eliminate important tail-risk information."

In [19]:
q1 = df_clean["loan_amount"].quantile(0.25)
q3 = df_clean["loan_amount"].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outliers = df_clean[
    (df_clean["loan_amount"] < lower) |
    (df_clean["loan_amount"] > upper)
]

print("IQR bounds:", lower, upper)
print("\nFlagged observations:")
print(outliers[["customer_id", "loan_amount", "income", "asset_value"]])

IQR bounds: -1250.0 70750.0

Flagged observations:
   customer_id  loan_amount    income  asset_value
7          108       900000  110000.0       250000


# 12. Build financially meaningful ratios

After validating inputs, derived ratios can be very informative.

Examples:

- **Loan-to-value (LTV)** = loan amount / asset value
- **Debt-to-income style measure** = debt payment / income
- **Loan-to-income** = loan amount / income

Always inspect the result after computing it.

### What to say

> "After creating a derived feature, I treat it like a new data source: I check missingness, distribution, and whether the values are economically plausible."

In [20]:
df_clean["ltv"] = np.where(
    df_clean["asset_value"] > 0,
    df_clean["loan_amount"] / df_clean["asset_value"],
    np.nan
)

df_clean["payment_to_income"] = np.where(
    df_clean["income"] > 0,
    df_clean["debt_payment"] / df_clean["income"],
    np.nan
)

print(df_clean[
    ["customer_id", "loan_to_income", "ltv", "payment_to_income"]
].describe())

       customer_id  loan_to_income        ltv  payment_to_income
count     10.00000        7.000000  10.000000           9.000000
mean     105.50000        1.523678   0.565297           0.014104
std        3.02765        2.936720   1.066855           0.003958
min      101.00000        0.333333   0.166667           0.006098
25%      103.25000        0.368512   0.216896           0.012195
50%      105.50000        0.416667   0.224747           0.013684
75%      107.75000        0.498452   0.250000           0.016667
max      110.00000        8.181818   3.600000           0.019512


In [21]:
print("Largest LTV values:")
print(
    df_clean.nlargest(5, "ltv")[
        ["customer_id", "loan_amount", "asset_value", "ltv"]
    ]
)

Largest LTV values:
   customer_id  loan_amount  asset_value       ltv
7          108       900000       250000  3.600000
3          104        45000       150000  0.300000
1          102        30000       120000  0.250000
6          107        50000       200000  0.250000
2          103        25000       110000  0.227273


# 13. Verify cleaning after every major transformation

This is where many candidates stop too early.

After cleaning, repeat the important diagnostics.

### Minimum verification checklist

- row count
- duplicates
- missing counts
- data types
- infinities
- summary statistics
- min/max for critical columns
- manually inspect rows that were changed

### What to say

> "I want to re-run the same diagnostics after cleaning. That lets me confirm that I fixed the intended problem without creating a new one."

In [22]:
print("=== SHAPE ===")
print(df_clean.shape)

print("\n=== DUPLICATES ===")
print(df_clean.duplicated().sum())

print("\n=== DTYPES ===")
print(df_clean.dtypes)

print("\n=== MISSING VALUES ===")
print(df_clean.isna().sum())

print("\n=== INFINITE NUMERIC VALUES ===")
numeric = df_clean.select_dtypes(include=np.number)
print(np.isinf(numeric).sum())

print("\n=== SUMMARY ===")
print(df_clean.describe().T)

=== SHAPE ===
(10, 13)

=== DUPLICATES ===
0

=== DTYPES ===
customer_id                   int64
date                 datetime64[us]
income                      float64
loan_amount                   int64
asset_value                   int64
debt_payment                float64
utilization                 float64
credit_score                float64
default                       int64
loan_to_income              float64
income_missing                int64
ltv                         float64
payment_to_income           float64
dtype: object

=== MISSING VALUES ===
customer_id          0
date                 0
income               0
loan_amount          0
asset_value          0
debt_payment         1
utilization          1
credit_score         1
default              0
loan_to_income       3
income_missing       0
ltv                  0
payment_to_income    1
dtype: int64

=== INFINITE NUMERIC VALUES ===
customer_id          0
income               0
loan_amount          0
asset_value        

# 14. Compare before vs after cleaning

A model-risk reviewer should be able to explain what changed.

Useful checks:

- rows removed
- values modified
- distributions shifted
- number of missing values before and after
- whether unaffected columns remained unchanged

### What to say

> "I would compare the pre- and post-cleaning datasets so the transformation is auditable. I want to know exactly how many observations were removed or modified and why."

In [23]:
print("Original rows:", len(df))
print("Clean rows   :", len(df_clean))

print("\nOriginal missing values:")
print(df.isna().sum())

print("\nClean missing values:")
print(df_clean.isna().sum())

Original rows: 11
Clean rows   : 10

Original missing values:
customer_id     0
date            0
income          1
loan_amount     0
asset_value     0
debt_payment    1
utilization     1
credit_score    1
default         0
dtype: int64

Clean missing values:
customer_id          0
date                 0
income               0
loan_amount          0
asset_value          0
debt_payment         1
utilization          1
credit_score         1
default              0
loan_to_income       3
income_missing       0
ltv                  0
payment_to_income    1
dtype: int64


# 15. Leakage check before modeling

A clean feature can still be invalid for modeling if it contains future information.

For a default model, suspicious fields might include:

- recovery amount
- collections activity after default
- days since default
- charge-off status recorded after prediction date

### What to say

> "Before modeling, I want to verify that every predictor would have been available at the time the prediction is supposed to be made. Otherwise, we may have target leakage even if the data is technically clean."

In [24]:
# Example only — imagine these columns existed:
potential_leakage = [
    "recovery_amount",
    "days_since_default",
    "post_default_collection_status"
]

print("Potential leakage fields to challenge:")
for col in potential_leakage:
    print("-", col)

Potential leakage fields to challenge:
- recovery_amount
- days_since_default
- post_default_collection_status


# 16. Final reusable inspection function

In an interview, a compact helper can save time if you are comfortable writing it correctly.

Do not over-engineer. Simple, readable code is safer.

In [25]:
def inspect_data(df):
    print("Shape:", df.shape)
    print("\nDtypes:")
    print(df.dtypes)

    print("\nMissing values:")
    print(df.isna().sum())

    print("\nDuplicates:", df.duplicated().sum())

    print("\nSummary:")
    print(df.describe(include="all"))

inspect_data(df_clean)

Shape: (10, 13)

Dtypes:
customer_id                   int64
date                 datetime64[us]
income                      float64
loan_amount                   int64
asset_value                   int64
debt_payment                float64
utilization                 float64
credit_score                float64
default                       int64
loan_to_income              float64
income_missing                int64
ltv                         float64
payment_to_income           float64
dtype: object

Missing values:
customer_id          0
date                 0
income               0
loan_amount          0
asset_value          0
debt_payment         1
utilization          1
credit_score         1
default              0
loan_to_income       3
income_missing       0
ltv                  0
payment_to_income    1
dtype: int64

Duplicates: 0

Summary:
       customer_id                 date         income   loan_amount    asset_value  debt_payment  utilization  credit_score    default  \


# 17. Interview decision tree

Use this mentally during the screen.

## A. Missing value

Ask:

**Is the variable ordered through time?**

- yes → consider `ffill()` / interpolation if justified
- no → consider median / mean / mode / row removal

Then ask:

**Could missingness itself carry information?**

If yes, consider preserving a missing indicator.

---

## B. Huge value

Do **not** immediately scale.

Check:

1. zero denominator?
2. unit mismatch?
3. percentage vs decimal?
4. parsing error?
5. genuine extreme observation?

Only after deciding the value is legitimate should you think about scaling.

---

## C. Outlier

Ask:

**Data error or genuine tail observation?**

- error → correct / mark missing / exclude with justification
- genuine → generally retain unless modeling methodology specifically requires treatment

---

## D. Ratio

Before computing:

- validate denominator
- check units

After computing:

- check missingness
- check infinities
- check min/max
- inspect top and bottom observations

---

## E. Cleaning completed

Always re-run:

```python
df.shape
df.isna().sum()
df.duplicated().sum()
df.describe()
np.isinf(df.select_dtypes("number")).sum()
```

# 18. Compact script to memorize

This is the workflow you should be able to reproduce quickly from memory.

In [ ]:
# 1. Load
df = pd.read_csv("data.csv")

# 2. Inspect
print(df.head())
print(df.shape)
print(df.dtypes)
print(df.isna().sum())
print(df.describe(include="all"))
print("Duplicates:", df.duplicated().sum())

# 3. Clean obvious structural issues
df = df.drop_duplicates()

# Example type conversion
# df["income"] = pd.to_numeric(df["income"], errors="coerce")

# 4. Check invalid / suspicious values
# print(df[df["income"] <= 0])
# print(df.nlargest(10, "loan_amount"))

# 5. Handle missing values only after understanding the variable
# df["income"] = df["income"].fillna(df["income"].median())
# df["x"] = df["x"].ffill()
# df["x"] = df["x"].interpolate()

# 6. Derived ratio: validate denominator first
# df.loc[df["income"] <= 0, "income"] = np.nan
# df["loan_ratio"] = df["loan_amount"] / df["income"]

# 7. Verify AFTER cleaning
print(df.shape)
print(df.isna().sum())
print(df.describe())
print("Duplicates:", df.duplicated().sum())

numeric = df.select_dtypes(include=np.number)
print("Infinite values:")
print(np.isinf(numeric).sum())

# 19. High-value phrases to use in the interview

You do not need to say all of these. Use them naturally when relevant.

> **"Before fixing the value, I want to understand whether it is a data-quality error or a genuine extreme observation."**

> **"Scaling changes the representation of the data; it does not correct an invalid observation."**

> **"I want to validate the denominator before computing the ratio."**

> **"The imputation method introduces an assumption, so I want that assumption to match the structure of the variable."**

> **"I'll compare summary statistics before and after cleaning to make sure the transformation did what I intended."**

> **"I would not automatically remove an outlier because genuine tail observations may be important for risk modeling."**

> **"I want to verify that every feature would have been available at prediction time so we do not introduce leakage."**

> **"After creating a derived variable, I'll sanity-check its distribution and inspect the extreme values."**

# 20. Final checklist before moving to modeling

You should be able to answer **yes** to all of these:

- [ ] I understand the unit of observation.
- [ ] I know the target variable.
- [ ] Important columns have correct data types.
- [ ] I checked missingness.
- [ ] I checked exact and business-key duplicates.
- [ ] I checked impossible values.
- [ ] I investigated suspicious outliers.
- [ ] I validated denominators before computing ratios.
- [ ] I justified each imputation method.
- [ ] I checked for unit inconsistencies.
- [ ] I verified the cleaned dataset again.
- [ ] I checked for leakage.
- [ ] My code runs from top to bottom.

---

## Next step

After this workflow becomes automatic, the natural next module is:

**Part 3 — Logistic Regression for the JPMorgan screen**

That should include:

- model intuition
- exact sklearn syntax
- train/test split
- scaling
- classification metrics
- class imbalance
- coefficient interpretation
- leakage traps
- a complete runnable credit-default example